# Records — 01: RECORDS-type collection without STAC

**Persona:** Metadata librarian / catalogue operator.

**Goal:** Run a pure OGC API — Records catalogue (no STAC fields, no geometry)
through DynaStore's generic storage pipeline.  Demonstrates that the role-based
driver architecture supports metadata-only catalogues by setting
``ItemsPostgresqlDriverConfig.collection_type="RECORDS"`` — the PG driver
then skips the geometry sidecar automatically, and the Records extension
serves ``/records`` endpoints against the same storage row.

## What's different from STAC

- `collection_type="RECORDS"` → no geometry column, no GeometrySidecar DDL.
- Only `item_metadata` + `attributes` sidecars — records carry title, description,
  keywords, and arbitrary typed attributes.
- Served under `/records/...` endpoints (see `extensions/records/records_service.py`).
- No `/stac/*` conformance URIs on this collection.

## Prerequisites

- DynaStore is running and reachable at `DYNASTORE_BASE_URL`.
- Admin token in `DYNASTORE_ADMIN_TOKEN` (or `DYNASTORE_SYSADMIN_TOKEN`).
- The `records` extension is loaded (scope_catalog / api_open / …).

In [ ]:
import os
import json
import uuid as _uuid
import time as _t

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL    = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
ADMIN_TOKEN = (
    os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or ""
)

headers = {"Authorization": f"Bearer {ADMIN_TOKEN}"} if ADMIN_TOKEN else {}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60.0)

CATALOG_ID    = f"rec01-{_uuid.uuid4().hex[:8]}"
COLLECTION_ID = "thesaurus-descriptors"
print(f"Using catalog={CATALOG_ID} collection={COLLECTION_ID}")


In [ ]:
# Bootstrap: create ephemeral catalog (via STAC catalog endpoint — STAC serves
# as the catalog-creation API; individual collections can then live under it
# with any collection_type).

payload = {
    "id": CATALOG_ID,
    "type": "Catalog",
    "title": "Records demo catalog",
    "description": "Harvested metadata records — no STAC features.",
    "stac_version": "1.1.0",
    "conformsTo": [],
    "links": [],
}
for _attempt in range(3):
    r = client.post("/stac/catalogs", content=json.dumps(payload))
    if r.status_code in (200, 201, 409):
        break
    _t.sleep(1)
assert r.status_code in (200, 201, 409), r.text[:400]
print("Catalog created" if r.status_code != 409 else "Catalog exists")


## Step 1 — Create a RECORDS-type collection

Collection creation is where we opt out of geometry.  The storage driver
config carries ``collection_type="RECORDS"``; the PG driver's
``_effective_sidecars`` helper sees this and injects only the sidecars
relevant for record-only collections (`item_metadata` + `attributes`).


In [ ]:
# Create a RECORDS collection via /records endpoint (the records extension
# registers its own POST handler for collection creation with RECORDS-typed
# storage pre-wired).

rec_collection = {
    "id": COLLECTION_ID,
    "type": "Collection",
    "title": "Thesaurus descriptors",
    "description": "Controlled-vocabulary records; no spatial component.",
    "keywords": ["thesaurus", "records", "metadata"],
    "license": "CC-BY-4.0",
}
r = client.post(f"/catalogs/{CATALOG_ID}/collections", content=json.dumps(rec_collection))
# Falls back to STAC's collection-creation if records-specific route is absent.
if r.status_code >= 400:
    r = client.post(
        f"/stac/catalogs/{CATALOG_ID}/collections",
        content=json.dumps({**rec_collection, "stac_version": "1.1.0", "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [[None, None]]}}, "links": []}),
    )
assert r.status_code in (200, 201), r.text[:400]
print("Collection created")


In [ ]:
# Step 2 — Pin the storage driver config to RECORDS mode.
# This is what removes geometry from the pipeline for this collection.

driver_cfg = {
    "class_key": "ItemsPostgresqlDriverConfig",
    "collection_type": "RECORDS",
    "sidecars": [
        {"sidecar_type": "item_metadata"},
        {"sidecar_type": "attributes"},
    ],
}
r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/ItemsPostgresqlDriverConfig",
    content=json.dumps(driver_cfg),
)
assert r.status_code in (200, 201, 204), r.text[:400]
print("Driver config pinned: collection_type=RECORDS")


## Step 2 — Route WRITE + READ through PostgreSQL

A minimal routing config: PG for both WRITE and READ, no fan-out, no INDEX, no BACKUP.
Notice the use of the canonical class name `ItemsPostgresqlDriver` (post-Phase-1 rename)
— legacy routing entries with `CollectionPostgresqlDriver` still validate through the
`config_rewriter` primitive, but new deployments pin to the canonical name.

In [ ]:
routing_cfg = {
    "class_key": "CollectionRoutingConfig",
    "operations": {
        "WRITE": [{"driver_id": "ItemsPostgresqlDriver", "on_failure": "fatal"}],
        "READ":  [{"driver_id": "ItemsPostgresqlDriver"}],
    },
}
r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/CollectionRoutingConfig",
    content=json.dumps(routing_cfg),
)
assert r.status_code in (200, 201, 204), r.text[:400]
print("Routing pinned: WRITE/READ → ItemsPostgresqlDriver")


## Step 3 — Write a few records

Records are `Feature(geometry=None, properties={…})` as far as the storage pipeline
is concerned.  The PG driver skips the geometry sidecar for RECORDS collections,
so the insert path only touches the hub + item_metadata + attributes tables.

In [ ]:
records = [
    {
        "type": "Feature",
        "id": f"desc-{i:03d}",
        "geometry": None,
        "properties": {
            "title": f"Descriptor {i}",
            "description": "Example thesaurus entry",
            "broader": f"root-cat-{i % 3}",
            "language": "en",
            "keyword_count": i,
        },
    }
    for i in range(1, 6)
]
payload = {"type": "FeatureCollection", "features": records}
r = client.post(
    f"/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    content=json.dumps(payload),
)
assert r.status_code in (200, 201, 204), r.text[:400]
print(f"Inserted {len(records)} records")


## Step 4 — Read back via `/records` (no STAC envelope)

The Records extension returns OGC API — Records resource shapes (`type: Record`),
which carry only the OGC Records core + record-core conformance classes; no STAC
`assets`, no `stac_extensions`.

In [ ]:
r = client.get(f"/records/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items?limit=3")
if r.status_code == 404:
    # Some deployments only expose records endpoints under a different prefix.
    r = client.get(f"/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items?limit=3")
assert r.status_code == 200, r.text[:400]
out = r.json()
print(f"Returned {len(out.get('features', []))} records; first:")
print(json.dumps(out.get("features", [])[:1], indent=2)[:1000])


## Teardown

Keeps the per-collection configs in place but removes the ephemeral catalog
(and its collections) so re-running the notebook starts clean.

In [ ]:
for path in [
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
    f"/stac/catalogs/{CATALOG_ID}",
]:
    r = client.delete(path)
    print(f"DELETE {path} → {r.status_code}")
client.close()
